<a href="https://colab.research.google.com/github/mekaeli/ELE767_CODES/blob/main/Laboratoire_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Segment de codes 1
def nom_de_la_fonction(argument1, argument2 =" valeur par defaut "):
    pass

"""
NOM_DE_LA_FONCTION # en majuscule
breve description de la fonction
@Arguments :
argument1 { type voulu pour l’argument1 } : description de ce que doit etre l
’argument1
argument2 { type voulu pour l’argument2 } : description de ce que doit etre l
’argument2
# ( facultatif ) explication du choix pour la valeur par defaut
@Return :
{ type de l’objet retourne par la fonction } : description de l’objet
retourne par la fonction
# Si d’autres objets sont retournes , la meme syntaxe est utilisee pour
chaque objet .
( facultatif ) Note : Information qui peut etre importante a connaitre pour l’
utilisation de la fonction
"""


In [ ]:
# Segment de codes 2
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import KFold
# Pour l’affichage des figures sans plt . show ()
%matplotlib inline

In [ ]:
# Segment de codes 3
# Commande originale (Colab/Linux) :
# !ls -lah

from pathlib import Path
import shutil
import subprocess

if shutil.which("ls"):
    subprocess.run("ls -lah", shell=True, check=False)
else:
    # Fallback Windows (ou environnements sans `ls`)
    for p in sorted(Path('.').iterdir()):
        if p.is_dir():
            print(f"{p.name}/")
        else:
            print(f"{p.name}\t{p.stat().st_size} bytes")


In [ ]:
# Segment de codes 4
# Transfert des datasets vers des variables ( DataFrames )
data_ds=pd.read_csv("data.csv")

In [ ]:
# Segment de codes 5
print (" __data__ ")
print ( data_ds . info ())
print ( data_ds . head ())
print ( data_ds . tail ())




In [ ]:
# Segment de codes 6
def create_sets( data_ds , n_fold =5) :
  """
  CREATE_SETS
  Transforme les bases de donnees afin d’etre compatible avec l’ entrainement
  du modele avec la methode " fit "
  @Arguments :
  data_ds {pd. DataFrame } : Base de donnees d’ entrainement
  n_fold { int } : nombre de division pour le kfold
  @Return :
  { tensorflow . BatchDataset } : Base de donnees d’ entrainement
  { tensorflow . BatchDataset } : Base de donnees de test
  """
  # Melanger les valeurs de la base de donnees
  data = data_ds . sample ( frac =1) . reset_index ( drop = True )

  # Diviser les variables et les resultats
  data_output = data . pop("number")

  # Extraire les indexes pour le kfold une seule fois
  for train_index , test_index in KFold ( n_fold ). split ( data ) :
    # Creer la base de donnees d’ entrainement
    X_train = ( data . values )[ train_index ]
    y_train = ( data_output . values )[ train_index ]

    ### Creer la base de donnees de test
    X_test = ( data . values )[ test_index ]
    y_test = ( data_output . values )[ test_index ]
    break

  # Transformer les bases de donnees pour etre utilisees directement dans fit
  # Les donnees sont melangees une autre fois avant d’etre retournees
  train_set = tf. data . Dataset . from_tensor_slices (( X_train , y_train ))
  train_set = train_set . shuffle ( len ( X_train ))
  train_set = train_set . batch (1)

  test_set = tf. data . Dataset . from_tensor_slices (( X_test , y_test ))
  test_set = test_set . shuffle (len ( X_test ))
  test_set = test_set . batch (1)

  return train_set, test_set

In [ ]:
# Segment de codes 7
# Creer un modele sequentiel
perceptron = tf.keras.Sequential([
  layers.Dense(4, activation='sigmoid', dtype='float64')
])
# Compiler le modele pour l’entrainement
perceptron.compile(optimizer='SGD',
                loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                metrics=['accuracy'])

In [ ]:
def training(model, data_ds, epochs=10, n_fold=5, verbose_train=1, verbose_test=1):
    """
    TRAINING

    Fonction servant a entrainer un model et en verifiant les capacites du modele sur les donnees de test

    @Arguments :
        model {tensorflow.python.keras.engine.sequential.Sequential} : Modele   tensorflow a entrainer
        data_ds {pd.DataFrame} : Base de données d'entrainement
        epochs {int} : Nombre d'epoques a faire pour l'entrainement du model
        n_fold {int} : Nombre de fold pour le kfold
        verbose_train {int} : 1 pour afficher l'avancement de l'apprentissage,  0 aucun affichage
        verbose_test {int} : 1 pour afficher l'avancement de l'evaluation, 0 aucun affichage

    @Return :
        {dict} :'train_accuracy' {list} : precision du modele sur la base d'entrainement
        'train_loss' {list} : perte du modele sur la base d'entrainement
        'test_accuracy' {list} : precision du modele sur la base de test
        'test_loss' {list} : perte du modele sur la base de test
    """
    # Initialisation des listes recevants
    train_accuracy = []
    train_loss = []
    test_accuracy = []
    test_loss = []

    # Creer les bases de donnees pour l'entrainement
    train_set, test_set = create_sets(data_ds, n_fold)

    # Evaluer le modele avant l'entrainement
    result = model.evaluate(train_set, verbose=verbose_train)
    train_accuracy.append(result[1])
    train_loss.append(result[0])

    result = model.evaluate(test_set, verbose=verbose_test)
    test_accuracy.append(result[1])
    test_loss.append(result[0])

    # Boucle d'entrainement
    for e in range(epochs):
        print('Epoch #{}'.format(e + 1))

        result = model.fit(train_set, epochs=1, verbose=verbose_train)
        train_accuracy.append(result.history['accuracy'][-1])
        train_loss.append(result.history['loss'][-1])

        result = model.evaluate(test_set, verbose=verbose_test)
        test_accuracy.append(result[1])
        test_loss.append(result[0])

    return {
        'train_accuracy': train_accuracy,
        'train_loss': train_loss,
        'test_accuracy': test_accuracy,
        'test_loss': test_loss
    }

In [ ]:
# Segment de codes 9
def figure (training_dict , suptitle , figsize =(12 ,6)):
    """
    FIGURE
    Produit une figure avec les statistiques de la fonction 'training '
    @Arguments :
    training_dict { dict ( list )} : Dictionnaire retournee par la fonction
    training
    figsize { Tuple } : Dimension de la figure
    """
    # Demarre une figure avec la taille (en pouce ) donner en arguments
    plt.figure(figsize = figsize , facecolor ='w')

    # Titre global de la figure
    plt . suptitle (suptitle)

    # Premiere sous - figure -> Precision du modele durant l' entrainement
    plt . subplot (121)
    plt . title ('Precision du modele')
    plt . xlabel ('Nombre d\' epoques')
    plt . ylabel ('Precision de la classification')
    # Produit une courbe pour chaque base de donnees
    plt . plot ( training_dict ['train_accuracy'], 'r', label ='training')
    plt . plot ( training_dict ['test_accuracy'], 'y', label ='test')
    # Limite de la precision pour rendre les valeurs entre 0% et 100% visibles
    plt . ylim (0 ,1.01)
    # Produire un grille en arriere - plan
    # plt . grid (True , which =' major ', color = '#666666 ' , linestyle ='-')
    plt . grid ()
    plt . minorticks_on ()
    plt . grid (True , which ='minor', color='#999999', linestyle ='-', alpha =0.2)
    plt . legend () # Afficher les labels des courbes

    # Deuxieme sous - figure -> Resultat de la fonction de perte durant l'entrainement
    plt . subplot (122)
    plt . title ('Perte du modele')
    plt . xlabel ('Nombre d\' epoques')
    plt . ylabel ('Resultat de la fonction de perte')
    # Produire une courbe pour chaque base de donnees
    plt . plot ( training_dict ['train_loss'],'r', label ='training')
    plt . plot ( training_dict ['test_loss'],'y', label ='test');
    # S'assurer que la valeur minimum de l'ordonnee soit 0
    plt . ylim (0) ;
    # Produire un grille en arriere - plan
    # plt . grid (True , which =' major ', color = '#666666 ' , linestyle ='-')
    plt . grid ()
    plt . minorticks_on ()
    plt . grid (True , which ='minor', color='#999999', linestyle ='-', alpha =0.2)
    plt . legend ()
    return

In [ ]:
# Segment de codes 10
result = training(perceptron, data_ds, epochs=25)

In [ ]:
# Segment de codes 11
figure(result, " Apprentissage d’un perceptron pour la classification des       signauxsonores")


In [ ]:
# Segment de codes 12
perceptron.save('modeles/perceptron.keras')

In [ ]:
# Segment de codes 13

#création d’un MLP à deux couches
mlp2c = tf.keras.Sequential([
    layers.Dense(512, activation='sigmoid', dtype='float64'),
    layers.Dense(4, activation='sigmoid', dtype='float64')
])

# Compiler le modele pour l’entrainement
mlp2c.compile(optimizer='SGD',
             loss=tf.keras.losses.SparseCategoricalCrossentropy(),
             metrics=['accuracy'])

# Entraîner le modèle (MLP à deux couches)
result_mlp2c = training(mlp2c, data_ds, epochs=25)

# Vérification de la structure du modele
mlp2c.summary()

# Afficher les statistiques du modèle
figure(result_mlp2c, "Apprentissage d’un MLP à deux couches")

# Enregistrer le modèle
mlp2c.save('modeles/mlp2c.keras')

In [ ]:
# Segment de codes 14
# Creer une couche d’entree
# La dimension des Tensors utilises est de (None , 1, 1560)
model_input = layers.Input(shape =( None, 1, data_ds.shape [1] -1))

# Creer les couches du reseau de neurones
x = layers.Dense (512, activation ='sigmoid',dtype ='float64')(model_input )
model = layers . Dense (4, activation ='sigmoid', dtype ='float64')(x)

# Valider le modele
model = tf.keras.Model(model_input, model)

# Compiler le modele pour l’ entrainement
model.compile (optimizer ='SGD',
              loss = tf.keras.losses.SparseCategoricalCrossentropy() ,
              metrics =['accuracy'])

# vérification de la steucture du modele
model.summary()

In [ ]:
# Segment de codes 15

# Création d’un MLP à deux couches avec abandon
mlp2cd = tf.keras.Sequential([
    layers.Dense(512, activation='sigmoid', dtype='float64'),
    layers.Dropout(0.2, dtype='float64'),
    layers.Dense(4, activation='sigmoid', dtype='float64')
])

# Compiler le modele pour l’entrainement
mlp2cd.compile(optimizer='SGD',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

# Entraîner le modèle (MLP à deux couches avec abandon)
result_mlp2cd = training(mlp2cd, data_ds, epochs=25)

# Vérification de la structure du modele
mlp2cd.summary()

# Afficher les statistiques du modèle
figure(result_mlp2cd, "Apprentissage d’un MLP à deux couches avec abandon")

# Enregistrer le modèle
mlp2cd.save('modeles/mlp2cd.keras')

In [ ]:
# Segment de codes 16
# Création d’un MLP à trois couches sans abandon
mlp3c = tf.keras.Sequential([
    layers.Dense(512, activation='relu', dtype='float64'),
    layers.Dense(128, activation='relu', dtype='float64'),
    layers.Dense(4, dtype='float64'),
    layers.Softmax(dtype='float64')
])

# Compiler le modele pour l’entrainement
mlp3c.compile(optimizer='SGD',
             loss=tf.keras.losses.SparseCategoricalCrossentropy(),
             metrics=['accuracy'])

# Entraîner le modèle (MLP à trois couches)
result_mlp3c = training(mlp3c, data_ds, epochs=25)

# Vérification de la structure du modele
mlp3c.summary()

# Afficher les statistiques du modèle
figure(result_mlp3c, "Apprentissage d’un MLP à trois couches")

# Enregistrer le modèle
mlp3c.save('modeles/mlp3c.keras')

In [ ]:
# Segment de codes 16 (suite) — 3.13 (modèle profond au choix)

# 1) Présentation (modèle choisi)
# - Le modèle choisi est un réseau récurrent de type LSTM.
# - L’implémentation est ci-dessous dans `build_lstm_model(...)`.

# 2) Références bibliographiques
# - Hochreiter & Schmidhuber (1997) — Long Short-Term Memory.
# (Les détails sont dans la cellule Markdown juste au-dessus.)

# 3) Expérimentation (même data.csv, k-fold k=5)

# 3a) Préparation des données (X, y) à partir de data.csv
#     - X: toutes les colonnes sauf 'number'
#     - y: la colonne 'number' (classe)

# Cellule autonome (imports + chargement data.csv si nécessaire)
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import KFold
from tensorflow.keras import layers

if "data_ds" not in globals():
    data_ds = pd.read_csv("data.csv")

# Hyperparamètres (ajustables si nécessaire)
N_FOLDS = 5
EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
RANDOM_STATE = 42

# Préparer X (features) et y (labels) à partir de data_ds
feature_cols = [c for c in data_ds.columns if c != "number"]
X_all = data_ds[feature_cols].to_numpy(dtype=np.float32)
y_all = data_ds["number"].to_numpy(dtype=np.int64)

n_features = X_all.shape[1]
n_classes = int(np.max(y_all) + 1)

# Interprétation "séquence" : (timesteps=n_features, channels=1)
X_all_seq = X_all.reshape((-1, n_features, 1))

def build_lstm_model(n_features: int, n_classes: int) -> tf.keras.Model:
    """Construit et compile un modèle LSTM pour la classification."""
    model = tf.keras.Sequential([
        layers.Input(shape=(n_features, 1), dtype="float32"),
        layers.LSTM(64),
        layers.Dense(64, activation="relu"),
        layers.Dense(n_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return model

# Afficher la structure (un exemple de modèle)
_model_preview = build_lstm_model(n_features, n_classes)
_model_preview.summary()

# 3b) Entraînement/évaluation en k-fold avec k=5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

fold_metrics = []
best_fold_acc = -1.0
best_fold = None
save_path = Path("modeles") / "deep_313_lstm.keras"
save_path.parent.mkdir(parents=True, exist_ok=True)

for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X_all_seq), start=1):
    print(f"\n===== Fold {fold_idx}/{N_FOLDS} =====")
    X_train, y_train = X_all_seq[train_idx], y_all[train_idx]
    X_test, y_test = X_all_seq[test_idx], y_all[test_idx]

    shuffle_buf = min(len(train_idx), 10_000)
    train_set = (
        tf.data.Dataset.from_tensor_slices((X_train, y_train))
        .shuffle(shuffle_buf)
        .batch(BATCH_SIZE)
    )
    test_set = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE)

    model_fold = build_lstm_model(n_features, n_classes)
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        restore_best_weights=True,
    )
    model_fold.fit(
        train_set,
        validation_data=test_set,
        epochs=EPOCHS,
        verbose=0,
        callbacks=[early_stop],
    )

    test_loss, test_acc = model_fold.evaluate(test_set, verbose=0)
    fold_metrics.append({
        "fold": fold_idx,
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
    })
    print(f"Test loss={test_loss:.4f} | Test accuracy={test_acc:.4f}")

    # 3d) Sauvegarde du modèle (meilleur fold)
    # Sauvegarder immédiatement si c'est le meilleur fold
    if float(test_acc) > best_fold_acc:
        best_fold_acc = float(test_acc)
        best_fold = fold_idx
        model_fold.save(save_path)
        print(f"Nouveau meilleur modèle sauvegardé: {save_path}")

# 3c) Affichage des résultats pour chaque fold + moyenne
metrics_df = pd.DataFrame(fold_metrics)
display(metrics_df)

mean_loss = metrics_df["test_loss"].mean()
mean_acc = metrics_df["test_accuracy"].mean()

print("\nMoyennes sur 5 folds :")
print(f"Loss moyenne  : {mean_loss:.4f}")
print(f"Accuracy moy. : {mean_acc:.4f}")

print(f"\nMeilleur fold: {best_fold} (accuracy={best_fold_acc:.4f})")
print(f"Modèle final sauvegardé : {save_path}")

# 4 — Déterminer la classe des segments audios mystères

Une nouvelle base de données `mystere.csv` est fournie pour simuler la mise en production d’un modèle d’IA prédisant cette base de données.

- `mystere.csv` **ne contient pas** la colonne `number`.
- Objectif : utiliser les modèles entraînés pour **prédire** la classe de chaque ligne de `mystere.csv`.

Segments à réaliser (selon l’énoncé) :
- **Segment 17** : Charger la base mystère et créer un `Dataset`.
- **Segment 18** : Charger le perceptron et prédire.
- **Segment 19** : Prédire avec tous les modèles et afficher le résultat commun.
- **Segment 20** : Exemple d’utilisation de la méthode `add` (ajout d’une couche `Softmax`).

In [ ]:
# Segment de codes 17 — Chargement de la base de données mystères
import numpy as np
import pandas as pd
import tensorflow as tf

# Charger data.csv (pour récupérer l'ordre des features)
data_ds = pd.read_csv("data.csv")
feature_cols = [c for c in data_ds.columns if c != "number"]

# Charger mystere.csv (ne contient pas 'number')
mystere_ds = pd.read_csv("mystere.csv")
print("__mystere__")
print(mystere_ds.info())

# Aligner les colonnes de mystere.csv sur celles de data.csv
X_mystere = mystere_ds.copy()
if set(X_mystere.columns) == set(feature_cols):
    X_mystere = X_mystere[feature_cols]
elif X_mystere.shape[1] == len(feature_cols):
    X_mystere.columns = feature_cols
else:
    raise ValueError(
        f"Colonnes incompatibles: mystere.csv a {X_mystere.shape[1]} colonnes, "
        f"mais data.csv a {len(feature_cols)} colonnes de features."
    )

# Créer le Dataset TensorFlow
mystere_set = tf.data.Dataset.from_tensor_slices(X_mystere.to_numpy(dtype=np.float32))
mystere_set = mystere_set.batch(1)

__mystere__
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Columns: 1560 entries, x0_MS_0 to W59_ED
dtypes: float64(1560)
memory usage: 61.1 KB
None


In [ ]:
# Segment de codes 18 — Chargement du perceptron entraîné et prédiction
import numpy as np
import tensorflow as tf

model = tf.keras.models.load_model("modeles/perceptron.keras")
y_pred_perceptron = np.argmax(model.predict(mystere_set, verbose=0), axis=1)

print("Prédictions du perceptron (aperçu) :")
print(y_pred_perceptron[:10])

Prédictions du perceptron (aperçu) :
[2 2 1 2 0]


In [ ]:
# Segment de codes 19 — Prédire le résultat des données avec tous les modèles
import numpy as np
from scipy import stats
import tensorflow as tf

# Liste des modèles du laboratoire (selon l'énoncé)
model_paths = [
    "modeles/perceptron.keras",
    "modeles/mlp2c.keras",
    "modeles/mlp2cd.keras",
    "modeles/mlp3c.keras",
]

result = []
for model_path in model_paths:
    model = tf.keras.models.load_model(model_path)
    prediction = model.predict(mystere_set, verbose=0)
    # Retourne l'index ayant la plus grande valeur
    result.append(np.argmax(prediction, axis=1))

# Transformer la liste en array numpy (shape: nb_modeles x nb_exemples)
result = np.array(result)

# Afficher les résultats
print("Résultat de tous les modèles :")
print(result)

# Résultat commun entre les modèles (mode par colonne)
mode_res = stats.mode(result, axis=0, keepdims=False)
common_pred = mode_res.mode
print("Résultat commun entre les modèles :")
print(common_pred)

Résultat de tous les modèles :
[[2 2 1 2 0]
 [2 2 1 2 0]
 [2 2 1 2 0]
 [2 2 1 2 0]]
Résultat commun entre les modèles :
[2 2 1 2 0]


In [ ]:
# Segment de codes 20 — Utilisation de la méthode "add"
import tensorflow as tf
from tensorflow.keras import layers

# Exemple : construire un perceptron en utilisant `add` (Dense + Softmax)
perceptron_add = tf.keras.Sequential()
perceptron_add.add(layers.Dense(4, dtype='float64'))
perceptron_add.add(layers.Softmax(dtype='float64'))

# Compiler le modèle pour l’entraînement
perceptron_add.compile(
    optimizer='SGD',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'],
)

# (Optionnel) Afficher la structure
perceptron_add.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax (Softmax)               │ ?                      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)